# Lab: Vector Search with Azure AI Search

In this lab you search a hotels index **by meaning instead of by keywords**.

1. Turn your question into a vector (a list of 1,536 numbers) with an Azure OpenAI **embedding** model.
2. Ask Azure AI Search for the hotels whose `DescriptionVector` is **closest** to your vector (cosine similarity).
3. Compare pure **vector** search with **hybrid** (keyword + vector) search.

```
your question ──► Azure OpenAI (text-embedding-ada-002) ──► query vector
                                                              │
                     Azure AI Search index ◄──────────────────┘
                     (vector-search-quickstart, field: DescriptionVector)
                              │
                              ▼
                     top-k most similar hotels
```

**How you sign in:** with your university account (Identikey) through Microsoft Entra ID. You don't need any API keys.

## 1. Install packages

Run this once. In Google Colab, it can take about 30 seconds.

In [1]:
%pip install -q azure-search-documents azure-identity openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Your instructor gives you these values. They aren't secrets, since access is controlled by your account.
If a `.env` file exists next to this notebook, its values take priority.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # optional: reads a .env file if one exists

AZURE_TENANT_ID        = os.getenv("AZURE_TENANT_ID", "<your-university-tenant-id>")
AZURE_SEARCH_ENDPOINT  = os.getenv("AZURE_SEARCH_ENDPOINT", "https://5350-lab.search.windows.net")
AZURE_SEARCH_INDEX     = os.getenv("AZURE_SEARCH_INDEX", "vector-search-quickstart")
AZURE_OPENAI_ENDPOINT  = os.getenv("AZURE_OPENAI_ENDPOINT", "https://5350.openai.azure.com")
EMBEDDING_DEPLOYMENT   = os.getenv("EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
VECTOR_FIELD           = os.getenv("VECTOR_FIELD", "DescriptionVector")

# Optional, instructors only: if set, that service uses the key instead of your Identikey.
# Students should leave these UNSET.
AZURE_SEARCH_KEY = os.getenv("AZURE_SEARCH_KEY") or None
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY") or None

SEARCH_AUTH = "API key" if AZURE_SEARCH_KEY else "Entra ID (Identikey)"
OPENAI_AUTH = "API key" if AZURE_OPENAI_KEY else "Entra ID (Identikey)"

print("Tenant :", AZURE_TENANT_ID)
print("Search :", AZURE_SEARCH_ENDPOINT, "/", AZURE_SEARCH_INDEX, "  auth =", SEARCH_AUTH)
print("OpenAI :", AZURE_OPENAI_ENDPOINT, "/", EMBEDDING_DEPLOYMENT, "  auth =", OPENAI_AUTH)
if AZURE_SEARCH_KEY or AZURE_OPENAI_KEY:
    print("\n⚠️  An API key is set, so role assignments are NOT being tested for that service.\n"
          "   To test as a student, remove AZURE_SEARCH_KEY / AZURE_OPENAI_KEY from .env and restart the kernel.")

Tenant : 3ded8b1b-070d-4629-82e4-c0b019f46057
Search : https://5350-lab.search.windows.net / vector-search-quickstart   auth = Entra ID (Identikey)
OpenAI : https://5350.openai.azure.com / text-embedding-ada-002   auth = Entra ID (Identikey)


## 3. Sign in

When you run this cell, it prints a **code** and a link (https://microsoft.com/devicelogin).
Open the link, paste the code, and sign in with your **Identikey**. This works in Colab, Jupyter, and VS Code.

In [3]:
from azure.identity import DeviceCodeCredential, AzureCliCredential, ChainedTokenCredential

credential = None
if not (AZURE_SEARCH_KEY and AZURE_OPENAI_KEY):
    tenant = None if AZURE_TENANT_ID.startswith("<") else AZURE_TENANT_ID
    credential = ChainedTokenCredential(
        AzureCliCredential(tenant_id=tenant),          # used if you ran `az login` on your laptop
        DeviceCodeCredential(tenant_id=tenant),        # otherwise: prints a code to sign in
    )
    # Request one token now so the sign-in happens here, not in the middle of a search.
    credential.get_token("https://search.azure.com/.default")
    print("✅ Signed in with Entra ID")
else:
    print("✅ Using API keys for both services (no sign-in needed)")

✅ Signed in with Entra ID


## 4. Create the clients

In [4]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX,
    credential=AzureKeyCredential(AZURE_SEARCH_KEY) if AZURE_SEARCH_KEY else credential,
)

if AZURE_OPENAI_KEY:
    openai_client = AzureOpenAI(azure_endpoint=AZURE_OPENAI_ENDPOINT,
                                api_key=AZURE_OPENAI_KEY,
                                api_version="2024-10-21")
else:
    openai_client = AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        azure_ad_token_provider=get_bearer_token_provider(
            credential, "https://cognitiveservices.azure.com/.default"),
        api_version="2024-10-21",
    )
print("✅ Clients ready")

✅ Clients ready


## 5. Check your access

This cell tests each service on its own. If something fails, it tells you **which permission is missing**.
Show the output to your instructor.

In [5]:
from azure.core.exceptions import HttpResponseError, ClientAuthenticationError
import openai

ok = True

# 1) Azure OpenAI: can we create an embedding?
try:
    v = openai_client.embeddings.create(input="test", model=EMBEDDING_DEPLOYMENT).data[0].embedding
    print(f"✅ Azure OpenAI embeddings OK ({len(v)} dimensions)   [{OPENAI_AUTH}]")
except (openai.AuthenticationError, openai.PermissionDeniedError) as e:
    ok = False
    print(f"❌ Azure OpenAI: {e.status_code}   [{OPENAI_AUTH}]")
    if AZURE_OPENAI_KEY:
        print("   The API key is wrong or was rotated. Remove AZURE_OPENAI_KEY from .env to use your Identikey.")
    else:
        print("   Your account is missing the 'Cognitive Services OpenAI User' role (or 'Azure AI User') on the\n"
              "   OpenAI resource, or the role hasn't synced yet (up to 1 hour). Ask your instructor.")
    print("   Details:", e.message[:300])
except openai.NotFoundError as e:
    ok = False
    print(f"❌ Azure OpenAI: deployment '{EMBEDDING_DEPLOYMENT}' not found. Check EMBEDDING_DEPLOYMENT.\n   ", e.message[:300])

# 2) Azure AI Search: can we read the index?
try:
    n = search_client.get_document_count()
    print(f"✅ Azure AI Search OK ({n} documents in '{AZURE_SEARCH_INDEX}')   [{SEARCH_AUTH}]")
except (ClientAuthenticationError, HttpResponseError) as e:
    ok = False
    code = getattr(e, "status_code", None)
    print(f"❌ Azure AI Search: {code}   [{SEARCH_AUTH}]")
    if code == 404:
        print(f"   Index '{AZURE_SEARCH_INDEX}' not found.")
    elif AZURE_SEARCH_KEY:
        print("   The API key is wrong. Remove AZURE_SEARCH_KEY from .env to use your Identikey.")
    elif code == 401:
        print("   The search service only accepts API keys. Instructor: Search service → Settings → Keys →\n"
              "   API access control → 'Both'.")
    else:
        print("   Your account is missing the 'Search Index Data Reader' role on the search service,\n"
              "   or the role hasn't synced yet (up to 1 hour). Ask your instructor.")

print("\nAll good. Continue to the next section." if ok else "\nFix the ❌ items above before continuing.")

✅ Azure OpenAI embeddings OK (1536 dimensions)   [Entra ID (Identikey)]
✅ Azure AI Search OK (7 documents in 'vector-search-quickstart')   [Entra ID (Identikey)]

All good. Continue to the next section.


## 6. Helpers

In [6]:
from azure.search.documents.models import VectorizedQuery

def embed(text: str) -> list[float]:
    # Turn text into a 1,536-number vector with Azure OpenAI.
    return openai_client.embeddings.create(input=text, model=EMBEDDING_DEPLOYMENT).data[0].embedding

def show(results, title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")
    count = 0
    for i, r in enumerate(results, 1):
        count = i
        desc = (r.get("Description") or "")[:160]
        print(f"\n#{i}  {r.get('HotelName')}  ({r.get('Category')})   score={r['@search.score']:.4f}")
        print(f"    {desc}...")
    if count == 0:
        print("No results.")

# Peek at a vector
v = embed("luxury hotel with spa and ocean view")
print(f"A query becomes {len(v)} numbers. First five: {[round(x, 4) for x in v[:5]]}")

A query becomes 1536 numbers. First five: [0.0034, 0.0068, 0.0009, -0.0188, 0.0017]


## 7. Pure vector search

There's no keyword matching here (`search_text=None`). The service returns the `k` hotels whose description vectors
are nearest to the query vector. Note that the query words don't need to appear in the description.

In [7]:
query = "luxury hotel with spa and ocean view"

vector_query = VectorizedQuery(
    vector=embed(query),
    k_nearest_neighbors=5,
    fields=VECTOR_FIELD,
)

results = search_client.search(
    search_text=None,                 # vector only
    vector_queries=[vector_query],
    select=["HotelName", "Description", "Category"],
    top=5,
)
show(results, f'Vector search: "{query}"')


Vector search: "luxury hotel with spa and ocean view"

#1  Swirling Currents Hotel  (Suite)   score=0.5109
    Spacious rooms, glamorous suites and residences, rooftop pool, walking access to shopping, dining, entertainment and the city center. Each room comes equipped w...

#2  Stay-Kay City Hotel  (Boutique)   score=0.5074
    This classic hotel is fully-refurbished and ideally located on the main commercial artery of the city in the heart of New York. A few minutes away is Times Squa...

#3  Gastronomic Landscape Hotel  (Suite)   score=0.5074
    The Gastronomic Hotel stands out for its culinary excellence under the management of William Dough, who advises on and oversees all of the Hotel’s restaurant se...

#4  Sublime Palace Hotel  (Boutique)   score=0.5032
    Sublime Palace Hotel is located in the heart of the historic center of Sublime in an extremely vibrant and lively area within short walking distance to the site...

#5  Old Century Hotel  (Boutique)   score=0.5012
    The 

## 8. Hybrid search (keyword + vector)

This sends **both** a keyword query and a vector query. Azure AI Search merges the two ranked lists with
*Reciprocal Rank Fusion (RRF)*. Hybrid search usually gives the best results in practice.

In [8]:
query = "family-friendly resort with kids activities"

results = search_client.search(
    search_text=query,                # keyword part
    vector_queries=[VectorizedQuery(vector=embed(query), k_nearest_neighbors=50, fields=VECTOR_FIELD)],
    select=["HotelName", "Description", "Category"],
    top=5,
)
show(results, f'Hybrid search: "{query}"')


Hybrid search: "family-friendly resort with kids activities"

#1  Swirling Currents Hotel  (Suite)   score=0.0325
    Spacious rooms, glamorous suites and residences, rooftop pool, walking access to shopping, dining, entertainment and the city center. Each room comes equipped w...

#2  Sublime Palace Hotel  (Boutique)   score=0.0323
    Sublime Palace Hotel is located in the heart of the historic center of Sublime in an extremely vibrant and lively area within short walking distance to the site...

#3  Old Century Hotel  (Boutique)   score=0.0320
    The hotel is situated in a nineteenth century plaza, which has been expanded and renovated to the highest architectural standards to create a modern, functional...

#4  Luxury Lion Resort  (Luxury)   score=0.0318
    Unmatched Luxury. Visit our downtown hotel to indulge in luxury accommodations. Moments from the stadium and transportation hubs, we feature the best in conveni...

#5  Stay-Kay City Hotel  (Boutique)   score=0.0164
    This 

## 9. Try it yourself

1. Write a query that uses **none** of the words in the description you want to find (for example, "place to relax by the water"). Does vector search still find relevant hotels? Does keyword-only search (`search_text=query`, no `vector_queries`) find them?
2. Change `k_nearest_neighbors` to 1, then 50. How do the results change?
3. Try a misspelled query ("luxary hotell spa"). Which search handles it better, keyword or vector?

In [9]:
my_query = "place to relax by the water"

show(search_client.search(search_text=my_query, select=["HotelName", "Description", "Category"], top=5),
     f'Keyword only: "{my_query}"')

show(search_client.search(search_text=None,
                          vector_queries=[VectorizedQuery(vector=embed(my_query), k_nearest_neighbors=5, fields=VECTOR_FIELD)],
                          select=["HotelName", "Description", "Category"], top=5),
     f'Vector only: "{my_query}"')


Keyword only: "place to relax by the water"

#1  Sublime Palace Hotel  (Boutique)   score=74.5293
    Sublime Palace Hotel is located in the heart of the historic center of Sublime in an extremely vibrant and lively area within short walking distance to the site...

#2  Old Century Hotel  (Boutique)   score=22.5631
    The hotel is situated in a nineteenth century plaza, which has been expanded and renovated to the highest architectural standards to create a modern, functional...

#3  Nordick's Valley Motel  (Boutique)   score=22.5631
    Only 90 miles (about 2 hours) from the nation's capital and nearby most everything the historic valley has to offer. Hiking? Wine Tasting? Exploring the caverns...

#4  Luxury Lion Resort  (Luxury)   score=21.6606
    Unmatched Luxury. Visit our downtown hotel to indulge in luxury accommodations. Moments from the stadium and transportation hubs, we feature the best in conveni...

#5  Swirling Currents Hotel  (Suite)   score=15.0520
    Spacious rooms